# Collaborative filtering

A user has rated some films. What should we show next when we know nothing about film genres, actors, or descriptions?

Collaborative filtering answers with the behavior of other users. There are two direct ways to do that:

- **User-based collaborative filtering** finds users with similar rating histories and recommends films those neighbors liked.
- **Item-based collaborative filtering** finds films with similar audiences and recommends films related to the ones this user liked.

We will use MovieLens 100K to build both neighborhood recommenders, hide each user's latest rating, and compare them with simple baselines. The only information they use is the user-item rating table.

## A catalogue described by its users

### a. Download the data

MovieLens contains explicit feedback: a user gives a film a rating from 1 to 5. There are no rows for films the user has not rated. That missingness is important: a zero in our matrix will mean **unknown**, not a rating of zero.

Collaborative filtering does not ask whether two films share a genre. It asks whether the same people rated them, or whether two people rated the same films in a similar way.

In [ ]:
import csv
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

DATASET_DIR = Path(".cache/ml-100k")
archive_path = DATASET_DIR.parent / "ml-100k.zip"

if not DATASET_DIR.exists():
    DATASET_DIR.parent.mkdir(exist_ok=True)
    if not archive_path.exists():
        print("Downloading MovieLens 100K...")
        urllib.request.urlretrieve(
            "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
            archive_path,
        )
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DATASET_DIR.parent)

The dataset is provided by the [GroupLens research group](https://grouplens.org/datasets/movielens/100k/). It contains 100,000 ratings from 943 users on 1,682 films.

Each row in `u.data` has four values: `user_id`, `item_id`, `rating`, and a timestamp. The timestamp gives us a way to make a realistic train/test split: the model must predict a later event without seeing it first.

### b. Turn the event table into a user-item matrix

We will use the matrix as a map of observed behavior. Rows are users, columns are films, and each known entry is a rating. The two neighborhood methods will look at this same table from opposite directions.

In [ ]:
with (DATASET_DIR / "u.item").open(encoding="latin-1", newline="") as file:
    item_titles = {
        int(row[0]): row[1]
        for row in csv.reader(file, delimiter="|")
    }

ratings = np.loadtxt(DATASET_DIR / "u.data", dtype=np.int64)
ratings = ratings[np.argsort(ratings[:, 3], kind="stable")]

USER_IDS = np.unique(ratings[:, 0])
ITEM_IDS = np.array(sorted(item_titles), dtype=int)
USER_INDEX = {int(user_id): index for index, user_id in enumerate(USER_IDS)}
ITEM_INDEX = {int(item_id): index for index, item_id in enumerate(ITEM_IDS)}

user_item = np.zeros((USER_IDS.size, ITEM_IDS.size), dtype=float)
for user_id, item_id, rating, _ in ratings:
    user_item[USER_INDEX[int(user_id)], ITEM_INDEX[int(item_id)]] = rating

known_ratings = user_item > 0
print(f"Users: {USER_IDS.size:,}")
print(f"Films: {ITEM_IDS.size:,}")
print(f"Ratings: {ratings.shape[0]:,}")
print(f"Matrix density: {known_ratings.mean():.2%}")

The matrix has one row per user and one column per film. A known rating is between 1 and 5; a zero means that the user has not rated that film.

The matrix is sparse, but it contains the evidence we need:

- To compare users, look across rows and keep only films both users rated.
- To compare films, look down columns and keep only users who rated both films.

No genre, title, actor, or description is used by either neighborhood method.

In [ ]:
sample_user_id = 1
sample_events = ratings[ratings[:, 0] == sample_user_id][:8]

print(f"First ratings from user {sample_user_id}:")
for _, item_id, rating, _ in sample_events:
    print(f"{int(rating)}/5  {item_titles[int(item_id)]}")

## Make the experiment honest

### a. Hide the latest rating

We sort all events by time and hide the last rating of every user. The recommenders see the earlier history and must predict the hidden event. This is a small chronological replay: the future rating is revealed only after the recommendation has been made.

The held-out rating is also useful for ranking evaluation. If it is 4 or 5, we call it a positive future event and check whether the film appears in the top 10.

In [ ]:
test_positions = []
for user_id in USER_IDS:
    user_positions = np.flatnonzero(ratings[:, 0] == user_id)
    test_positions.append(user_positions[-1])
test_positions = np.array(test_positions, dtype=int)

train_mask = np.ones(ratings.shape[0], dtype=bool)
train_mask[test_positions] = False
train_ratings = ratings[train_mask]
test_ratings = ratings[test_positions]

train_pairs = np.array(
    [
        [USER_INDEX[int(user_id)], ITEM_INDEX[int(item_id)], int(rating)]
        for user_id, item_id, rating, _ in train_ratings
    ],
    dtype=np.int64,
)
test_pairs = np.array(
    [
        [USER_INDEX[int(user_id)], ITEM_INDEX[int(item_id)], int(rating)]
        for user_id, item_id, rating, _ in test_ratings
    ],
    dtype=np.int64,
)

train_matrix = np.zeros_like(user_item)
for user_index, item_index, rating in train_pairs:
    train_matrix[user_index, item_index] = rating

positive_test_pairs = test_pairs[test_pairs[:, 2] >= 4]
print(f"Training ratings: {train_pairs.shape[0]:,}")
print(f"Held-out ratings: {test_pairs.shape[0]:,}")
print(f"Held-out positive ratings: {positive_test_pairs.shape[0]:,}")

### b. Start with popularity

A baseline gives the experiment a reference point. Here, popularity means the number of positive training ratings. It ignores the current user's taste, but popular films are often surprisingly difficult to beat.

In [ ]:
positive_counts = np.bincount(
    train_pairs[:, 1],
    weights=(train_pairs[:, 2] >= 4).astype(float),
    minlength=ITEM_IDS.size,
)
POPULAR_ITEM_INDICES = np.argsort(-positive_counts, kind="stable")

def recommend_popular(user_index, k=10):
    seen = train_matrix[user_index] > 0
    return [
        int(item_index)
        for item_index in POPULAR_ITEM_INDICES
        if not seen[item_index]
    ][:k]

popular_items = recommend_popular(USER_INDEX[sample_user_id], k=5)
print("Popular films not yet rated by the example user:")
for item_index in popular_items:
    print(item_titles[int(ITEM_IDS[item_index])])

## Find similar users

### a. User-based collaborative filtering

Imagine that user 1 and user 2 have both rated 20 films. If their ratings move together, they are neighbors. To predict a film for user 1, we look at those neighbors who have already rated it and transfer their rating deviations.

The steps are:

1. Center each user's ratings around that user's average, so generous and strict raters can still be compared.
2. Compute cosine similarity between rows using only observed ratings.
3. Keep the nearest users with enough films in common.
4. Predict an unseen film from the neighbors' weighted rating deviations.

This is **user-user collaborative filtering**: people who behaved similarly help one another.

In [ ]:
MIN_SHARED_RATINGS = 5
GLOBAL_MEAN = float(train_pairs[:, 2].mean())
known_train = train_matrix > 0
user_counts = known_train.sum(axis=1)
USER_MEANS = np.divide(
    train_matrix.sum(axis=1),
    user_counts,
    out=np.full(USER_IDS.size, GLOBAL_MEAN),
    where=user_counts > 0,
)

centered_users = (
    (train_matrix - USER_MEANS[:, np.newaxis]) * known_train
).astype(float)
user_norms = np.linalg.norm(centered_users, axis=1)
user_denominator = np.outer(user_norms, user_norms)
USER_SIMILARITY = np.divide(
    centered_users @ centered_users.T,
    user_denominator,
    out=np.zeros_like(user_denominator),
    where=user_denominator > 0,
)
USER_OVERLAP = known_train.astype(np.int32) @ known_train.astype(np.int32).T
USER_SIMILARITY[USER_OVERLAP < MIN_SHARED_RATINGS] = 0
np.fill_diagonal(USER_SIMILARITY, 0)


def user_based_scores(user_index, number_of_neighbors=40):
    similarities = USER_SIMILARITY[user_index]
    neighbor_indices = np.argsort(-similarities, kind="stable")[
        :number_of_neighbors
    ]
    weights = np.maximum(similarities[neighbor_indices], 0)
    neighbor_known = known_train[neighbor_indices]
    neighbor_deviations = (
        train_matrix[neighbor_indices]
        - USER_MEANS[neighbor_indices, np.newaxis]
    ) * neighbor_known
    numerator = weights @ neighbor_deviations
    denominator = weights @ neighbor_known

    scores = np.full(ITEM_IDS.size, USER_MEANS[user_index])
    valid = denominator > 0
    scores[valid] += numerator[valid] / denominator[valid]
    return scores


print(f"Users: {USER_IDS.size:,}")
print(f"User pairs with at least {MIN_SHARED_RATINGS} films in common: "
      f"{np.count_nonzero(USER_OVERLAP >= MIN_SHARED_RATINGS):,}")

The similarity matrix is a compact summary of the crowd. `USER_SIMILARITY[a, b]` is positive when users `a` and `b` tend to be above or below their own averages on the same films.

The prediction is deliberately local. A neighbor's rating matters only when that neighbor has rated the candidate film, and closer neighbors receive larger weights.

In [ ]:
def nearest_users(user_index, k=5):
    neighbor_indices = np.argsort(-USER_SIMILARITY[user_index], kind="stable")[:k]
    return [
        (
            int(USER_IDS[neighbor_index]),
            float(USER_SIMILARITY[user_index, neighbor_index]),
            int(USER_OVERLAP[user_index, neighbor_index]),
        )
        for neighbor_index in neighbor_indices
    ]


sample_user_index = USER_INDEX[sample_user_id]
print(f"Nearest users to user {sample_user_id}:")
for user_id, similarity, overlap in nearest_users(sample_user_index):
    print(f"user {user_id}: similarity={similarity:.3f}, "
          f"shared films={overlap}")

## Find similar films

### a. Item-based collaborative filtering

User neighborhoods can change whenever a new person arrives. Item neighborhoods are often easier to explain:

> People who rated this film also tended to rate those films.

For item-based filtering, we turn the matrix around. Two films are similar when many of the same users rated them in a similar direction around each film's average. To recommend for one user, we find films similar to the films already in that user's history and combine those local signals.

In [ ]:
item_counts = known_train.sum(axis=0)
ITEM_MEANS = np.divide(
    train_matrix.sum(axis=0),
    item_counts,
    out=np.full(ITEM_IDS.size, GLOBAL_MEAN),
    where=item_counts > 0,
)

centered_items = (
    (train_matrix - ITEM_MEANS[np.newaxis, :]) * known_train
).astype(float)
item_norms = np.linalg.norm(centered_items, axis=0)
item_denominator = np.outer(item_norms, item_norms)
ITEM_SIMILARITY = np.divide(
    centered_items.T @ centered_items,
    item_denominator,
    out=np.zeros_like(item_denominator),
    where=item_denominator > 0,
)
ITEM_OVERLAP = known_train.astype(np.int32).T @ known_train.astype(np.int32)
ITEM_SIMILARITY[ITEM_OVERLAP < MIN_SHARED_RATINGS] = 0
np.fill_diagonal(ITEM_SIMILARITY, 0)


def item_based_scores(user_index):
    rated_items = np.flatnonzero(known_train[user_index])
    if not rated_items.size:
        return ITEM_MEANS.copy()

    rating_deviations = (
        train_matrix[user_index, rated_items] - ITEM_MEANS[rated_items]
    )
    positive_similarities = np.maximum(
        ITEM_SIMILARITY[:, rated_items], 0
    )
    numerator = positive_similarities @ rating_deviations
    denominator = positive_similarities.sum(axis=1)

    scores = ITEM_MEANS.copy()
    valid = denominator > 0
    scores[valid] += numerator[valid] / denominator[valid]
    return scores


def recommend_from_scores(scores, user_index, k=10):
    scores = scores.copy()
    scores[known_train[user_index]] = -np.inf
    return np.argsort(-scores, kind="stable")[:k].astype(int).tolist()


def nearest_items(item_index, k=5):
    similar_indices = np.argsort(-ITEM_SIMILARITY[item_index], kind="stable")[:k]
    return [
        (
            int(ITEM_IDS[similar_index]),
            float(ITEM_SIMILARITY[item_index, similar_index]),
            int(ITEM_OVERLAP[item_index, similar_index]),
        )
        for similar_index in similar_indices
    ]


sample_item_index = ITEM_INDEX[int(sample_events[0, 1])]
print(f"Films similar to {item_titles[int(ITEM_IDS[sample_item_index])]}:")
for item_id, similarity, overlap in nearest_items(sample_item_index):
    print(f"{similarity:.3f}  {item_titles[item_id]} "
          f"(shared users={overlap})")

## Does neighborhood filtering help?

### a. Two questions, two metrics

We will measure two different things:

- **RMSE** asks how close the predicted rating is to the hidden rating.
- **HitRate@10** asks whether a hidden positive film appears in the ten recommendations.

We will compare four rules:

- `Random` ignores the ratings entirely.
- `Popular` recommends films with many positive ratings.
- `User-based` recommends from similar users.
- `Item-based` recommends from films similar to the user's history.

The comparison matters because a neighborhood method can predict ratings reasonably while still putting the wrong films at the top of the list.

In [ ]:
def recommend_random(user_index, rng, k=10):
    candidates = np.flatnonzero(~known_train[user_index])
    size = min(k, candidates.size)
    return rng.choice(candidates, size=size, replace=False).astype(int).tolist()


def hit_rate_at_k(recommendation_function, positive_pairs, k=10):
    hits = 0
    for user_index, item_index, _ in positive_pairs:
        recommendations = recommendation_function(int(user_index), k)
        hits += int(int(item_index) in recommendations)
    return hits / positive_pairs.shape[0]


evaluation_users = np.unique(test_pairs[:, 0]).astype(int)
user_score_by_user = {
    user_index: user_based_scores(user_index)
    for user_index in evaluation_users
}
item_score_by_user = {
    user_index: item_based_scores(user_index)
    for user_index in evaluation_users
}

user_predictions = np.array([
    np.clip(user_score_by_user[int(user_index)][int(item_index)], 1, 5)
    for user_index, item_index, _ in test_pairs
])
item_predictions = np.array([
    np.clip(item_score_by_user[int(user_index)][int(item_index)], 1, 5)
    for user_index, item_index, _ in test_pairs
])

global_rmse = np.sqrt(
    np.mean((test_pairs[:, 2] - GLOBAL_MEAN) ** 2)
)
user_rmse = np.sqrt(
    np.mean((test_pairs[:, 2] - user_predictions) ** 2)
)
item_rmse = np.sqrt(
    np.mean((test_pairs[:, 2] - item_predictions) ** 2)
)

random_rng = np.random.default_rng(21)


def random_recommendation(user_index, k=10):
    return recommend_random(user_index, random_rng, k)


def popular_recommendation(user_index, k=10):
    return recommend_popular(user_index, k)


def user_based_recommendation(user_index, k=10):
    return recommend_from_scores(user_score_by_user[user_index], user_index, k)


def item_based_recommendation(user_index, k=10):
    return recommend_from_scores(item_score_by_user[user_index], user_index, k)


hit_rates = {
    "Random": hit_rate_at_k(random_recommendation, positive_test_pairs),
    "Popular": hit_rate_at_k(popular_recommendation, positive_test_pairs),
    "User-based": hit_rate_at_k(
        user_based_recommendation, positive_test_pairs
    ),
    "Item-based": hit_rate_at_k(
        item_based_recommendation, positive_test_pairs
    ),
}

print(f"Held-out ratings evaluated: {test_pairs.shape[0]:,}")
print(f"Positive held-out ratings evaluated: "
      f"{positive_test_pairs.shape[0]:,}")
print(f"Held-out rating RMSE, global mean: {global_rmse:.3f}")
print(f"Held-out rating RMSE, user-based: {user_rmse:.3f}")
print(f"Held-out rating RMSE, item-based: {item_rmse:.3f}")
for method, hit_rate in hit_rates.items():
    print(f"{method:>12}: HitRate@10 = {hit_rate:.3f}")

plt.figure(figsize=(8, 5))
plt.bar(hit_rates.keys(), hit_rates.values())
plt.ylabel("HitRate@10")
plt.ylim(0, max(hit_rates.values()) * 1.25)
plt.title("Neighborhood recommenders on a chronological holdout")
plt.show()

The four numbers answer different questions. Random shows how hard it is to hit a future film by chance. Popularity shows what can be achieved without personalization. User-based filtering asks whether similar people help. Item-based filtering asks whether similar films help.

The two neighborhood methods may disagree. User-based filtering is intuitive when a person's taste matches a visible group of neighbors. Item-based filtering is often more stable and easier to explain because the recommendation can start from a film already in the user's history. Neither method is guaranteed to beat popularity on every split.

## What neighborhood collaborative filtering knows

Both methods learn only from ratings that already exist:

- User-based filtering needs users with overlapping histories. A brand-new user has no neighbors until they rate something.
- Item-based filtering needs films that have been rated by some users. A brand-new film has no item neighborhood until it receives feedback.
- Neither method reads genres, titles, actors, or descriptions. That makes them useful when metadata is missing, but creates a cold-start problem for new users and films.

The choice between the two is a choice about what should remain stable. People change and new people arrive; item relationships are often more reusable across users. The right choice depends on the product, the amount of overlap, and the evaluation results.

## Your turn

Try these changes one at a time:

- Change `MIN_SHARED_RATINGS` and see when similarities become reliable enough to use.
- Compare 5, 20, 40, and 100 neighbors in the user-based recommender.
- Remove mean-centering and observe how much generous or strict raters affect the result.
- Use Pearson correlation instead of cosine similarity.
- Compare user-based and item-based recommendations for a user with a short history.
- Hold out each user's last five ratings instead of only the last one.
- Add a minimum number of positive neighbors before allowing a recommendation.

The most useful exercise is to print the neighbors behind one recommendation. A score is easier to trust when you can explain which people or films contributed to it.

## In summary

- Collaborative filtering uses interaction patterns rather than item descriptions.
- User-based filtering finds people with similar rating histories and borrows their signals.
- Item-based filtering finds films rated by similar audiences and recommends them from the user's history.
- Mean-centering separates preference from a user's general rating generosity.
- A minimum overlap prevents a similarity based on only one shared film from dominating.
- A chronological holdout helps prevent future information from leaking into training.
- RMSE evaluates rating prediction, while HitRate@10 evaluates the recommendation list.
- Interaction history enables personalization but creates a cold-start problem for new users and films.